# NB03 — Data Analysis

**Question:** If I had invested in the S&P 500 ten years ago instead of leaving
the money in a savings account, what would the difference be today?

This notebook reads the tidy table produced by `NB02-Data-Transformation.py`
(`data/processed/monthly_returns.csv`) and explores it. It does not repeat the
collection or the preparation work.

Run this notebook from the repository root, like NB01 and NB02.


In [8]:
import pandas as pd
import plotly.graph_objects as go

# NOTE: not covered in course - plotly is used instead of matplotlib because the
# public page needs charts a reader can hover over to read exact monthly values.

df = pd.read_csv("../data/processed/monthly_returns.csv")

# One row per series per month -> pivot to one row per month, one column per series.
monthly = df.pivot(index="date", columns="series", values="return_pct")

print(monthly.shape)
print(monthly.index.min(), "to", monthly.index.max())
print(monthly.isna().sum())
monthly.head()

(120, 2)
2016-08 to 2026-07
series
BRMSA0104    0
SP500        0
dtype: int64


series,BRMSA0104,SP500
date,,
2016-08,0.006667,0.364226
2016-09,0.006667,-0.908904
2016-10,0.006667,-0.679893
2016-11,0.006667,1.024944
2016-12,0.006667,3.771080


Both series cover the same 120 months, August 2016 to July 2026, with no missing
values. One row is one calendar month; each column holds that month's return in
percent (S&P 500 from the price index, the savings account from the APY).


## 1. What the monthly returns look like

Before comparing totals, look at the spread. An average return hides how the two
options behave month to month, and that difference is half of the answer.


In [9]:
monthly.describe()

series,BRMSA0104,SP500
count,120.000000,120.000000
mean,0.018557,1.103098
std,0.015715,3.488425
min,0.005000,-19.068070
25%,0.007021,-0.225223
50%,0.008333,1.544856
75%,0.039750,3.171892
max,0.044167,8.220955


In [10]:
fig = go.Figure()

for name, col, color in [("Savings account", "BRMSA0104", "#e07b39"),
                         ("S&P 500", "SP500", "#1f77b4")]:
    fig.add_trace(go.Box(
        x=monthly[col], name=name, marker_color=color,
        boxpoints="all", jitter=0.6, pointpos=0,
        marker_size=5, marker_opacity=0.5,
        text=monthly.index,
        hovertemplate="%{text}: %{x:.2f}%<extra>" + name + "</extra>",
    ))

fig.add_annotation(x=monthly["SP500"].min(), y="S&P 500", text="Mar 2020",
                   showarrow=True, arrowhead=0, ax=0, ay=-32, font_size=12)

fig.update_layout(
    title="Monthly returns, Aug 2016 - Jul 2026 (each dot is one month)",
    template="plotly_white", height=300, showlegend=False, boxgap=0.35,
    margin=dict(l=125, r=30, t=65, b=55),
)
fig.update_xaxes(title_text="Monthly return (%)", zeroline=True,
                 zerolinecolor="#bbb", zerolinewidth=2)

fig.show()

One dot per month rather than a histogram: the S&P 500 has a single month near
-19% and nothing between -19% and -8%, so binned bars leave a wide empty gap that
reads as a broken chart. Plotting every month puts both series on one axis, and
the box adds the median and quartiles. Hovering a dot gives the month and its
exact return.

[Write here what you see. Concrete values from the table and chart above:
S&P 500 ranges from -19.07% (March 2020) to +8.22% (May 2025), standard deviation
3.49%, and 35 of the 120 months are negative. The savings account ranges from
0.005% to 0.044%, standard deviation 0.016%, and no month is negative.]
